# Arquitecturas actuales de modelos de generacion de video

**Notebook Colab comentado - Punto 4**

Este notebook ejemplifica, con implementaciones pequenas y ejecutables, las arquitecturas mas usadas hoy en generacion de video. No pretende entrenar un modelo comercial como Sora, Veo, Runway, Pika o Stable Video Diffusion; el objetivo es mostrar los componentes internos: representacion espacio-temporal, difusion en pixeles, difusion latente, U-Net 3D, Transformers de difusion, modelos secuencials, image-to-video y control/adapters.


## 0. Preparacion del entorno

Las demostraciones principales usan NumPy, Matplotlib y TensorFlow/Keras. La ultima seccion deja un ejemplo opcional con `diffusers` para inspeccionar un pipeline real en Colab con GPU.


In [ ]:
# Instalacion opcional en Colab para la ultima seccion.
# !pip install -q diffusers transformers accelerate safetensors imageio imageio-ffmpeg

import math
import numpy as np
import matplotlib.pyplot as plt

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    print('TensorFlow:', tf.__version__)
except Exception as exc:
    tf = None
    keras = None
    layers = None
    print('TensorFlow no disponible:', exc)

plt.rcParams['figure.figsize'] = (12, 3)


## 1. Utilidades: video sintetico y visualizacion

Para comparar arquitecturas usaremos un video artificial: un cuadrado que se mueve. Es simple, pero sirve para observar continuidad temporal, ruido, latentes y condicionamiento sin descargar datos externos.


In [ ]:
def make_moving_square_video(frames=16, height=64, width=64, square=12, direction='right'):
    """Devuelve un video con forma (frames, height, width, channels)."""
    video = np.zeros((frames, height, width, 1), dtype=np.float32)
    for t in range(frames):
        ratio = t / max(frames - 1, 1)
        if direction == 'right':
            x = int((width - square) * ratio)
            y = height // 2 - square // 2
        elif direction == 'down':
            x = width // 2 - square // 2
            y = int((height - square) * ratio)
        else:
            x = int((width - square) * ratio)
            y = int((height - square) * ratio)
        video[t, y:y + square, x:x + square, 0] = 1.0
    return video


def show_video_strip(video, title='', frames_to_show=8, cmap='gray'):
    """Muestra varios fotogramas de un video en una tira horizontal."""
    idxs = np.linspace(0, video.shape[0] - 1, frames_to_show, dtype=int)
    fig, axes = plt.subplots(1, len(idxs), figsize=(1.8 * len(idxs), 2.2))
    for ax, idx in zip(axes, idxs):
        frame = video[idx]
        if frame.shape[-1] == 1:
            ax.imshow(frame[..., 0], cmap=cmap, vmin=0, vmax=1)
        else:
            ax.imshow(np.clip(frame, 0, 1))
        ax.set_title(f't={idx}')
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

video = make_moving_square_video()
print('Forma del video:', video.shape)
show_video_strip(video, 'Video sintetico original')


## 2. Arquitectura A: difusion de video en espacio de pixeles

Los primeros modelos de video diffusion extendian DDPM directamente a tensores `(frames, alto, ancho, canales)`. La red aprende a predecir ruido en pixeles. Ventaja: trabaja sobre la senal visual directa. Desventaja: es costoso para alta resolucion y muchos fotogramas.

La muestra agrega ruido a todos los fotogramas y luego aplica un denoising visual muy simple para ilustrar la tarea que aprenderia la red.


In [ ]:
def add_ddpm_noise(x0, alpha_bar=0.35, seed=11):
    """x_t = sqrt(alpha_bar) x_0 + sqrt(1-alpha_bar) epsilon."""
    rng = np.random.default_rng(seed)
    eps = rng.normal(size=x0.shape).astype(np.float32)
    xt = math.sqrt(alpha_bar) * x0 + math.sqrt(1 - alpha_bar) * eps
    return xt.astype(np.float32), eps


def toy_pixel_denoise(noisy_video):
    """Denoiser no aprendido: suavizado temporal y clipping.
    Un modelo real reemplaza esto con una U-Net/Transformer entrenado.
    """
    padded = np.pad(noisy_video, ((1, 1), (0, 0), (0, 0), (0, 0)), mode='edge')
    smooth = (padded[:-2] + padded[1:-1] + padded[2:]) / 3.0
    return np.clip(smooth, 0, 1)

noisy_pixel, real_noise = add_ddpm_noise(video, alpha_bar=0.25)
denoised_pixel = toy_pixel_denoise(noisy_pixel)

show_video_strip(np.clip(noisy_pixel, 0, 1), 'A. Difusion en pixeles: video ruidoso')
show_video_strip(denoised_pixel, 'A. Muestra de denoising temporal simple')
print('Tensor ruidoso:', noisy_pixel.shape, '| Ruido que el modelo aprende a predecir:', real_noise.shape)


## 3. Arquitectura B: difusion latente de video

Stable Diffusion redujo el costo generando en un espacio latente. En video se hace algo parecido: un encoder comprime el clip, la difusion ocurre en latentes y un decoder reconstruye los fotogramas. Modelos como Stable Video Diffusion pertenecen a esta familia.

Aqui usamos un encoder/decoder didactico por pooling y upsampling. No es un VAE real, pero muestra el flujo arquitectonico.


In [ ]:
def encode_video_latent(x, factor=4):
    """Compresion espacial simple por promedio: (T,H,W,C) -> (T,H/f,W/f,C)."""
    t, h, w, c = x.shape
    x = x.reshape(t, h // factor, factor, w // factor, factor, c)
    return x.mean(axis=(2, 4))


def decode_video_latent(z, factor=4):
    """Decodificacion simple por nearest-neighbor."""
    return np.repeat(np.repeat(z, factor, axis=1), factor, axis=2)

latent = encode_video_latent(video, factor=4)
noisy_latent, _ = add_ddpm_noise(latent, alpha_bar=0.20, seed=21)
recon_from_noisy_latent = decode_video_latent(np.clip(noisy_latent, 0, 1), factor=4)
recon_clean = decode_video_latent(latent, factor=4)

print('Video original:', video.shape)
print('Latente comprimido:', latent.shape)
print('Reduccion aproximada de elementos:', video.size // latent.size, 'x')
show_video_strip(recon_clean, 'B. Video reconstruido desde latente limpio')
show_video_strip(recon_from_noisy_latent, 'B. Difusion latente: ruido en espacio comprimido')


## 4. Arquitectura C: U-Net espacio-temporal / 3D

Una U-Net de imagen mezcla informacion espacial con convoluciones 2D. Para video se agregan convoluciones 3D, atencion temporal o bloques temporales. La forma de entrada es `(batch, frames, alto, ancho, canales)`. La salida suele ser ruido predicho con la misma forma.


In [ ]:
if tf is not None:
    def build_tiny_video_unet(frames=16, height=64, width=64, channels=1):
        noisy_video = keras.Input(shape=(frames, height, width, channels), name='video_ruidoso')
        noise_level = keras.Input(shape=(1,), name='paso_difusion')

        # Embedding del paso de difusion. En modelos reales se usa sinusoidal + MLP.
        temb = layers.Dense(32, activation='swish')(noise_level)
        temb = layers.Dense(frames * height * width, activation='swish')(temb)
        temb = layers.Reshape((frames, height, width, 1))(temb)

        x = layers.Concatenate(axis=-1)([noisy_video, temb])
        s1 = layers.Conv3D(16, 3, padding='same', activation='swish')(x)
        s2 = layers.Conv3D(32, 3, strides=(1, 2, 2), padding='same', activation='swish')(s1)
        x = layers.Conv3D(64, 3, strides=(2, 2, 2), padding='same', activation='swish')(s2)
        x = layers.Conv3D(64, 3, padding='same', activation='swish')(x)
        x = layers.UpSampling3D((2, 2, 2))(x)
        x = layers.Concatenate()([x, s2])
        x = layers.Conv3D(32, 3, padding='same', activation='swish')(x)
        x = layers.UpSampling3D((1, 2, 2))(x)
        x = layers.Concatenate()([x, s1])
        x = layers.Conv3D(16, 3, padding='same', activation='swish')(x)
        pred_noise = layers.Conv3D(channels, 1, padding='same', name='ruido_predicho')(x)
        return keras.Model([noisy_video, noise_level], pred_noise)

    unet3d = build_tiny_video_unet()
    sample_video = noisy_pixel[None, ...]
    sample_t = np.array([[0.75]], dtype=np.float32)
    pred = unet3d([sample_video, sample_t], training=False)
    print('Entrada U-Net 3D:', sample_video.shape)
    print('Salida ruido predicho:', pred.shape)
    print('Parametros:', unet3d.count_params())
else:
    print('Ejecuta esta celda en Colab con TensorFlow para instanciar la U-Net 3D.')


## 5. Arquitectura D: Transformer de difusion para video (Video DiT)

Los modelos mas actuales representan el video como tokens o parches espacio-temporales. Sora describe imagenes y videos como colecciones de parches; CogVideoX usa un Transformer de difusion especializado. La idea es convertir el volumen de video en tokens, aplicar self-attention y reconstruir el volumen latente.


In [ ]:
def patchify_video(x, patch_t=2, patch_h=8, patch_w=8):
    """Convierte (T,H,W,C) en tokens de parches 3D."""
    t, h, w, c = x.shape
    x = x.reshape(t // patch_t, patch_t, h // patch_h, patch_h, w // patch_w, patch_w, c)
    x = x.transpose(0, 2, 4, 1, 3, 5, 6)
    tokens = x.reshape(-1, patch_t * patch_h * patch_w * c)
    grid = (t // patch_t, h // patch_h, w // patch_w)
    return tokens.astype(np.float32), grid


def unpatchify_video(tokens, grid, channels=1, patch_t=2, patch_h=8, patch_w=8):
    gt, gh, gw = grid
    x = tokens.reshape(gt, gh, gw, patch_t, patch_h, patch_w, channels)
    x = x.transpose(0, 3, 1, 4, 2, 5, 6)
    return x.reshape(gt * patch_t, gh * patch_h, gw * patch_w, channels)

tokens, grid = patchify_video(video)
print('Tokens de video:', tokens.shape, '| grid espacio-temporal:', grid)

if tf is not None:
    # Mini Video-DiT: proyeccion -> self-attention -> MLP -> reconstruccion de tokens.
    token_inputs = keras.Input(shape=(tokens.shape[0], tokens.shape[1]))
    x = layers.Dense(64)(token_inputs)
    x = layers.LayerNormalization()(x)
    x = layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
    x = layers.LayerNormalization()(x)
    x = layers.Dense(128, activation='gelu')(x)
    token_outputs = layers.Dense(tokens.shape[1])(x)
    tiny_dit = keras.Model(token_inputs, token_outputs)

    out_tokens = tiny_dit(tokens[None, ...], training=False).numpy()[0]
    out_video = unpatchify_video(out_tokens, grid)
    print('Salida del Transformer:', out_tokens.shape)
    show_video_strip(np.clip(out_video, 0, 1), 'D. Muestra reconstruida por mini Video-DiT sin entrenar')
else:
    print('Tokens listos para un Transformer de difusion.')


## 6. Arquitectura E: modelo secuencial o hibrido

Algunos sistemas generan video por pasos: dado el pasado, predicen el siguiente latente o bloque temporal. Esto es parecido a lenguaje, pero con tokens visuales. La ventaja es extender duracion; la desventaja es que puede acumular errores. Muchos modelos modernos combinan planificacion secuencial con refinamiento por difusion.


In [ ]:
def estimate_shift(prev_frame, next_frame):
    """Estimador didactico de movimiento usando centro de masa."""
    def center(frame):
        yy, xx = np.nonzero(frame[..., 0] > 0.5)
        if len(xx) == 0:
            return np.array([0.0, 0.0])
        return np.array([yy.mean(), xx.mean()])
    return center(next_frame) - center(prev_frame)


def shift_frame(frame, dy, dx):
    return np.roll(np.roll(frame, int(round(dy)), axis=0), int(round(dx)), axis=1)

# Usamos los primeros 4 frames como contexto y extrapolamos el movimiento.
context = video[:4]
shift = estimate_shift(context[-2], context[-1])
auto_frames = [f.copy() for f in context]
for _ in range(12):
    auto_frames.append(shift_frame(auto_frames[-1], shift[0], shift[1]))
auto_video = np.stack(auto_frames)

print('Movimiento estimado por frame (dy, dx):', shift)
show_video_strip(auto_video, 'E. Generacion secuencial didactica: extrapolar movimiento')


## 7. Arquitectura F: image-to-video condicionada

Modelos como Stable Video Diffusion son image-to-video: reciben una imagen inicial y generan movimiento. Arquitectonicamente, la condicion de imagen entra mediante un encoder visual y/o se concatena con latentes. La red no inventa toda la apariencia; anima una referencia.


In [ ]:
def image_to_video_conditioned(first_frame, frames=16, dx=3, dy=0):
    """Animacion deterministica para ilustrar condicion image-to-video."""
    generated = []
    frame = first_frame.copy()
    for t in range(frames):
        generated.append(frame)
        frame = shift_frame(frame, dy, dx)
    return np.stack(generated)

first_frame = video[0]
img2vid = image_to_video_conditioned(first_frame, frames=16, dx=3, dy=0)
show_video_strip(np.repeat(first_frame[None, ...], 8, axis=0), 'F. Condicion: una sola imagen inicial')
show_video_strip(img2vid, 'F. Muestra image-to-video: animar la condicion')


## 8. Arquitectura G: video controlado por trayectoria, pose o mapa estructural

Los modelos con ControlNet, adapters o condiciones estructurales reciben mapas de pose, profundidad, bordes, segmentacion o trayectorias. Estos controles guian la generacion y reducen ambiguedad.

La muestra usa una trayectoria 2D como condicion externa. En un modelo real, esa condicion se codifica y se inyecta en la U-Net/Transformer.


In [ ]:
def make_trajectory_control(frames=16, height=64, width=64):
    control = np.zeros((frames, height, width, 1), dtype=np.float32)
    for t in range(frames):
        x = int(8 + (width - 16) * t / (frames - 1))
        y = int(height / 2 + 16 * math.sin(2 * math.pi * t / frames))
        control[t, max(0, y-1):min(height, y+2), max(0, x-1):min(width, x+2), 0] = 1.0
    return control


def generate_from_control(control, square=10):
    frames, height, width, _ = control.shape
    out = np.zeros_like(control)
    for t in range(frames):
        yy, xx = np.nonzero(control[t, ..., 0] > 0.5)
        if len(xx):
            y, x = int(yy.mean()), int(xx.mean())
            out[t, max(0, y-square//2):min(height, y+square//2), max(0, x-square//2):min(width, x+square//2), 0] = 1.0
    return out

control = make_trajectory_control()
controlled_video = generate_from_control(control)
show_video_strip(control, 'G. Condicion estructural: trayectoria')
show_video_strip(controlled_video, 'G. Video generado siguiendo la trayectoria')


## 9. Arquitectura H: atencion temporal factorada

Para ahorrar memoria, muchos modelos alternan atencion espacial y temporal. La atencion temporal permite que una posicion del fotograma consulte la misma region en otros tiempos, ayudando a preservar identidad y movimiento.


In [ ]:
if tf is not None:
    class TemporalAttention(layers.Layer):
        def __init__(self, channels, heads=4):
            super().__init__()
            self.norm = layers.LayerNormalization()
            self.attn = layers.MultiHeadAttention(num_heads=heads, key_dim=max(channels // heads, 1))

        def call(self, x):
            # x: (batch, frames, height, width, channels)
            b = tf.shape(x)[0]
            f = tf.shape(x)[1]
            h = tf.shape(x)[2]
            w = tf.shape(x)[3]
            c = tf.shape(x)[4]
            seq = tf.transpose(x, [0, 2, 3, 1, 4])
            seq = tf.reshape(seq, [b * h * w, f, c])
            seq_norm = self.norm(seq)
            seq = seq + self.attn(seq_norm, seq_norm)
            seq = tf.reshape(seq, [b, h, w, f, c])
            return tf.transpose(seq, [0, 3, 1, 2, 4])

    sample_features = tf.random.normal((2, 16, 8, 8, 32))
    temporal_block = TemporalAttention(channels=32)
    attended = temporal_block(sample_features)
    print('Entrada:', sample_features.shape)
    print('Salida despues de atencion temporal:', attended.shape)
else:
    print('Ejecuta esta celda en Colab con TensorFlow para probar atencion temporal.')


## 10. Comparacion resumida de arquitecturas

| Arquitectura | Que genera | Ventaja | Limitacion |
|---|---|---|---|
| Difusion en pixeles | Video directamente en pixeles | Senal visual directa | Muy costosa |
| Difusion latente | Latentes de video | Eficiente y escalable | Depende de buen VAE/decoder |
| U-Net 3D | Ruido o latentes con convoluciones temporales | Buena estructura local | Menos global que Transformer |
| Video DiT | Tokens espacio-temporales | Escala con datos y computo | Atencion costosa |
| Secuencial/hibrido | Siguiente token/clip | Videos mas largos | Acumula errores |
| Image-to-video | Movimiento desde imagen inicial | Mayor control visual | Menos libertad creativa |
| Control/adapters | Video guiado por pose/profundidad/trayectoria | Control fino | Requiere condicion externa |


## 11. Pipeline real opcional con Diffusers

Esta celda no es necesaria para entender las arquitecturas, pero sirve para inspeccionar componentes de un modelo image-to-video real en Colab con GPU.


In [ ]:
# OPCIONAL EN COLAB CON GPU
# from diffusers import StableVideoDiffusionPipeline
# import torch
#
# pipe = StableVideoDiffusionPipeline.from_pretrained(
#     'stabilityai/stable-video-diffusion-img2vid-xt',
#     torch_dtype=torch.float16,
#     variant='fp16'
# ).to('cuda')
#
# print('Scheduler:', type(pipe.scheduler).__name__)
# print('VAE / decoder latente:', type(pipe.vae).__name__)
# print('Denoiser espacio-temporal:', type(pipe.unet).__name__)
# print('Encoder de imagen condicionante:', type(pipe.image_encoder).__name__)
#
# # Uso real:
# # frames = pipe(image_pil, num_frames=25, decode_chunk_size=8).frames[0]


## 12. Conclusion

Las arquitecturas actuales de generacion de video combinan las ideas de difusion vistas en imagenes con mecanismos temporales. La evolucion principal va desde U-Nets de difusion en pixeles hacia modelos latentes y Transformers de difusion sobre parches espacio-temporales. La clave es modelar simultaneamente apariencia, movimiento, condicionamiento textual y coherencia temporal.
